<a href="https://colab.research.google.com/github/susionoahmad/Bank-Branch-Operational-Analysis/blob/main/Membangun%20Early%20Warning%20System%20Kepuasan%20Nasabah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Membaca dataset operasional cabang yang baru kita buat
# (Nanti Anda bisa upload file CSV-nya ke Google Colab)
url = "https://raw.githubusercontent.com/datasets/investor-flow-usa/master/data/weekly.csv" # Contoh URL data sementara

print("Pandas berhasil di-import dan siap digunakan!")

Pandas berhasil di-import dan siap digunakan!


In [ ]:
# Membaca file CSV yang sudah di-upload ke Colab
df = pd.read_csv('dataset_operasional_cabang.csv')

# Menampilkan 5 baris pertama data untuk memastikan data terbaca
df.head()

,transaction_id,staff_name,service_type,arrival_time,start_time,end_time,satisfaction_score
0,TXN-1000,Citra (CS 1),Pembukaan Rekening,2026-05-07 11:28:00,2026-05-07 11:50:00,2026-05-07 12:30:00,2
1,TXN-1001,Budi (Teller 2),Transfer Antar Bank,2026-05-11 10:23:00,2026-05-11 10:28:00,2026-05-11 10:33:00,4
2,TXN-1002,Budi (Teller 2),Tarikan Tunai,2026-05-12 13:37:00,2026-05-12 14:12:00,2026-05-12 14:15:00,2
3,TXN-1003,Citra (CS 1),Pembukaan Rekening,2026-05-12 08:48:00,2026-05-12 08:59:00,2026-05-12 09:34:00,5
4,TXN-1004,Eka (CS 2),Komplain/CS,2026-05-15 13:50:00,2026-05-15 14:27:00,2026-05-15 14:45:00,2


In [ ]:
# Memfilter data hanya untuk transaksi 'Pembukaan Rekening'
data_cs = df[df['service_type'] == 'Pembukaan Rekening']

# Menampilkan 5 sampel teratas hasil filter
data_cs.head()

,transaction_id,staff_name,service_type,arrival_time,start_time,end_time,satisfaction_score
0,TXN-1000,Citra (CS 1),Pembukaan Rekening,2026-05-07 11:28:00,2026-05-07 11:50:00,2026-05-07 12:30:00,2
3,TXN-1003,Citra (CS 1),Pembukaan Rekening,2026-05-12 08:48:00,2026-05-12 08:59:00,2026-05-12 09:34:00,5
13,TXN-1013,Eka (CS 2),Pembukaan Rekening,2026-05-01 11:07:00,2026-05-01 11:40:00,2026-05-01 12:16:00,2
16,TXN-1016,Citra (CS 1),Pembukaan Rekening,2026-05-05 10:43:00,2026-05-05 10:47:00,2026-05-05 11:07:00,4
22,TXN-1022,Citra (CS 1),Pembukaan Rekening,2026-05-12 11:19:00,2026-05-12 11:56:00,2026-05-12 12:40:00,3


In [ ]:
# Menghitung jumlah nasabah (transaksi) untuk setiap tipe layanan
layanan_counts = df['service_type'].value_counts()

print("Total Nasabah per Tipe Layanan:")
print(layanan_counts)

Total Nasabah per Tipe Layanan:
service_type
Tarikan Tunai          109
Pembukaan Rekening     106
Komplain/CS            100
Transfer Antar Bank     97
Setoran Tunai           88
Name: count, dtype: int64


In [ ]:
# 1. Mengubah kolom teks menjadi tipe data DateTime resmi di Python
df['arrival_time'] = pd.to_datetime(df['arrival_time'])

# 2. Membuat kolom baru khusus angka jam (sama seperti EXTRACT HOUR di SQL)
df['jam_kedatangan'] = df['arrival_time'].dt.hour

# 3. Menampilkan data terbaru untuk melihat kolom 'jam_kedatangan' di paling kanan
df[['arrival_time', 'jam_kedatangan']].head()

,arrival_time,jam_kedatangan
0,2026-05-07 11:28:00,11
1,2026-05-11 10:23:00,10
2,2026-05-12 13:37:00,13
3,2026-05-12 08:48:00,8
4,2026-05-15 13:50:00,13


In [ ]:
# 1. Pastikan kolom start_time juga bertipe DateTime resmi
df['start_time'] = pd.to_datetime(df['start_time'])

# 2. Hitung selisih waktu tunggu (dalam satuan menit)
df['waktu_tunggu_menit'] = (df['start_time'] - df['arrival_time']).dt.total_seconds() / 60

# 3. Lakukan GROUP BY berdasarkan jam_kedatangan (Persis seperti SQL kemarin!)
rekap_per_jam = df.groupby('jam_kedatangan').agg(
    jumlah_nasabah=('transaction_id', 'count'),
    rata_rata_tunggu_menit=('waktu_tunggu_menit', 'mean'),
    skor_kepuasan=('satisfaction_score', 'mean')
).reset_index()

# 4. Tampilkan hasil rekapitulasi operasionalnya
rekap_per_jam

,jam_kedatangan,jumlah_nasabah,rata_rata_tunggu_menit,skor_kepuasan
0,8,85,6.658824,4.741176
1,9,63,6.730159,4.714286
2,10,77,6.272727,4.675325
3,11,86,25.732558,2.604651
4,12,58,27.086207,2.620690
5,13,60,27.850000,2.433333
6,14,71,6.957746,4.690141


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Mengubah skor kepuasan menjadi 2 kategori saja (Target Y)
# Jika skor >= 4 maka 1 (Puas), jika di bawah 4 maka 0 (Kecewa)
df['target_kepuasan'] = df['satisfaction_score'].apply(lambda x: 1 if x >= 4 else 0)

# 2. Mengubah teks 'service_type' menjadi kolom angka 0 dan 1 (Features X)
X = df[['waktu_tunggu_menit', 'service_type']]
X = pd.get_dummies(X, columns=['service_type'], drop_first=True)

# 3. Menentukan Target (Y)
y = df['target_kepuasan']

# 4. Menampilkan bentuk data X yang siap dipelajari oleh AI
X.head()

,waktu_tunggu_menit,service_type_Pembukaan Rekening,service_type_Setoran Tunai,service_type_Tarikan Tunai,service_type_Transfer Antar Bank
0,22.0,True,False,False,False
1,5.0,False,False,False,True
2,35.0,False,False,True,False
3,11.0,True,False,False,False
4,37.0,False,False,False,False


In [ ]:
# 1. Membagi data: 80% untuk latihan (Training) dan 20% untuk tes (Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Membuat objek AI bernama Decision Tree (Pohon Keputusan)
model = DecisionTreeClassifier(max_depth=3, random_state=42)

# 3. Proses TRANING (AI belajar dari data masa lalu cabang Bapak)
model.fit(X_train, y_train)

# 4. Proses TESTING (AI mencoba menebak kepuasan nasabah)
y_pred = model.predict(X_test)

# 5. Menghitung seberapa pintar AI kita (Akurasi)
akurasi = accuracy_score(y_test, y_pred)
print(f"🎯 Akurasi Model Machine Learning Anda: {akurasi * 100:.2f}%")

🎯 Akurasi Model Machine Learning Anda: 100.00%


In [ ]:
from sklearn.tree import export_text

# Melihat teks aturan keputusan yang dihasilkan oleh AI
tree_rules = export_text(model, feature_names=list(X.columns))
print(tree_rules)

|--- waktu_tunggu_menit <= 20.50
|   |--- class: 1
|--- waktu_tunggu_menit >  20.50
|   |--- class: 0



In [ ]:
import numpy as np

def prediksi_kepuasan_nasabah(waktu_tunggu, jenis_layanan):
    # 1. Membuat template baris data baru dengan struktur yang sama dengan variabel X
    input_data = pd.DataFrame(0, index=[0], columns=X.columns)

    # 2. Mengisi nilai waktu tunggu
    input_data['waktu_tunggu_menit'] = waktu_tunggu

    # 3. Mengisi nilai untuk One-Hot Encoding jenis layanan
    kolom_layanan = f'service_type_{jenis_layanan}'
    if kolom_layanan in input_data.columns:
        input_data[kolom_layanan] = 1 # Set True/1 untuk layanan yang dipilih

    # 4. Melakukan prediksi menggunakan model AI yang sudah dilatih
    prediksi = model.predict(input_data)[0]

    # 5. Mengubah hasil angka menjadi teks agar mudah dibaca
    status = "😊 PUAS (Skor 4-5)" if prediksi == 1 else "😞 KECEWA (Skor 1-3)"

    print(f"=== HASIL PREDIKSI AI ===")
    print(f"Layanan      : {jenis_layanan}")
    print(f"Waktu Tunggu : {waktu_tunggu} menit")
    print(f"Prediksi     : {status}")

# --- COBA UBAH ANGKA DI BAWAH INI UNTUK TES ---
prediksi_kepuasan_nasabah(waktu_tunggu=20.45, jenis_layanan="Pembukaan Rekening")

=== HASIL PREDIKSI AI ===
Layanan      : Pembukaan Rekening
Waktu Tunggu : 20.45 menit
Prediksi     : 😊 PUAS (Skor 4-5)
